In [ ]:
import os
import sys

print("Python is working!")
print("Python version:", sys.version)
print("Current working directory:", os.getcwd())

In [1]:
import sys
print(sys.version)

3.11.16 (main, Sep  2 2026, 23:33:55) [Clang 21.1.8 ]


In [2]:
from osgeo import gdal

print("GDAL version:", gdal.VersionInfo())

GDAL version: 3130300


Assignment 1

In [4]:
# Assignment 1

import os
import glob
import shutil
from osgeo import gdal

# File locations
source_directory = (
    "/Users/chrisarchibald/Desktop/GEOG 6805/"
    "Lab_3/LC08_L1TP_038032_20230621_20230630_02_T1"
)

output_directory = (
    "/Users/chrisarchibald/Desktop/GEOG 6805/"
    "Lab4_Assignment1"
)
os.makedirs(output_directory, exist_ok=True)

print("Source folder:", source_directory)
print("Output folder:", output_directory)

# Find TIFF files
img_list = (
    glob.glob(os.path.join(source_directory, "*.tif"))
    + glob.glob(os.path.join(source_directory, "*.TIF"))
    + glob.glob(os.path.join(source_directory, "*.tiff"))
    + glob.glob(os.path.join(source_directory, "*.TIFF"))
)

img_list = sorted(set(img_list))
print("Total TIFF files found:", len(img_list))

# Count 30m bands
count_30m = 0

# Check each file
for img in img_list:
    dataset = gdal.Open(img)

    if dataset is None:
        print("Could not open:", os.path.basename(img))
        continue

    geotransform = dataset.GetGeoTransform()
    x_resolution = abs(geotransform[1])
    y_resolution = abs(geotransform[5])
    filename = os.path.basename(img)

    print(
        filename,
        "- Resolution:",
        x_resolution, "x", y_resolution, "meters"
    )

    # Select 30m bands
    if (
        abs(x_resolution - 30) < 0.01
        and abs(y_resolution - 30) < 0.01
    ):
        # Create band folder
        img_name = os.path.splitext(filename)[0]
        new_folder = os.path.join(output_directory, img_name)
        os.makedirs(new_folder, exist_ok=True)

        destination = os.path.join(new_folder, filename)

        # Copy original file
        dataset = None
        shutil.copy2(img, destination)

        count_30m += 1
        print("Copied:", filename)

    else:
        # Skip other resolutions
        dataset = None
        print("Skipped:", filename)

# Final results
print("\nAssignment 1 completed!")
print("Number of 30m TIFF files organized:", count_30m)
print("Output folder:", output_directory)

Source folder: /Users/chrisarchibald/Desktop/GEOG 6805/Lab_3/LC08_L1TP_038032_20230621_20230630_02_T1
Output folder: /Users/chrisarchibald/Desktop/GEOG 6805/Lab4_Assignment1
Total TIFF files found: 18
LC08_L1TP_038032_20230621_20230630_02_T1_B1.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B1.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B10.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B10.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B11.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B11.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B2.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B2.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B3.TIF - Resolution: 30.0 x 30.0 meters


/Users/chrisarchibald/miniconda3/envs/geospatial/lib/python3.11/site-packages/osgeo/gdal.py:596: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B3.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B4.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B4.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B5.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B5.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B6.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B6.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B7.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B7.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B8.TIF - Resolution: 15.0 x 15.0 meters
Skipped: LC08_L1TP_038032_20230621_20230630_02_T1_B8.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_B9.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_20230630_02_T1_B9.TIF
LC08_L1TP_038032_20230621_20230630_02_T1_QA_PIXEL.TIF - Resolution: 30.0 x 30.0 meters
Copied: LC08_L1TP_038032_20230621_

Assignment #2

In [5]:

# Assignment 2


import ee
import geemap
import os
import glob

# Connect to Google Earth Engine
ee.Initialize()

print("Earth Engine initialized successfully!")


# Set folder paths
aoi_folder = "./AOI_Shapefiles"
download_folder = "./Lab4_Assignment2"

os.makedirs(download_folder, exist_ok=True)

# Find all shapefiles
aoi_files = glob.glob(os.path.join(aoi_folder, "*.shp"))
print("AOI shapefiles found:", len(aoi_files))

# Select Sentinel-2 bands originally at 10 m and 20 m
bands = [
    "B2", "B3", "B4", "B8",
    "B5", "B6", "B7", "B8A", "B11", "B12"
]

# Loop through each AOI
for aoi_file in aoi_files:

    aoi_name = os.path.splitext(os.path.basename(aoi_file))[0]
    print("\nProcessing AOI:", aoi_name)

    # Convert shapefile to Earth Engine geometry
    aoi = geemap.shp_to_ee(aoi_file)
    aoi_geometry = aoi.geometry()

    # Create AOI folder
    aoi_output = os.path.join(download_folder, aoi_name)
    os.makedirs(aoi_output, exist_ok=True)

    # Find matching Sentinel-2 images
    collection = (
        ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
        .filterBounds(aoi_geometry)
        .filterDate("2019-05-25", "2019-05-26")
        .filter(
            ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", 60)
        )
    )

    # Get image IDs
    image_ids = collection.aggregate_array("system:id").getInfo()
    print("Matching images found:", len(image_ids))

    # Download each image
    for image_id in image_ids:

        tile_name = image_id.split("/")[-1]
        print("\nDownloading:", tile_name)

        # Select required bands
        image = ee.Image(image_id).select(bands)

        # Create tile folder
        tile_folder = os.path.join(aoi_output, tile_name)
        os.makedirs(tile_folder, exist_ok=True)

        # Set output file path
        image_path = os.path.join(
            tile_folder, tile_name + ".tif"
        )

        # Skip existing downloads
        if os.path.exists(image_path):
            print("Already downloaded:", tile_name)
            continue

        # Download at 30 m resolution
        try:
            geemap.download_ee_image(
                image,
                image_path,
                region=aoi_geometry,
                crs="EPSG:4326",
                scale=30
            )
            print("Download completed:", image_path)

        except Exception as error:
            print("Download failed:", tile_name)
            print("Error:", error)

print("\nAssignment 2 completed!")
print("Output folder:", download_folder)


Earth Engine initialized successfully!
AOI shapefiles found: 3

Processing AOI: AOI_Tulsa
Matching images found: 0

Processing AOI: AOI_Nebe
Matching images found: 4

Downloading: 20190525T170859_20190525T171218_T14TPM


/Users/chrisarchibald/miniconda3/envs/geospatial/lib/python3.11/site-packages/geemap/common.py:12263: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


20190525T170859_20190525T171218_T14TPM:   0%|          |0/390 tiles [00:00<?]

Download completed: ./Lab4_Assignment2/AOI_Nebe/20190525T170859_20190525T171218_T14TPM/20190525T170859_20190525T171218_T14TPM.tif

Downloading: 20190525T170859_20190525T171218_T14TQM


20190525T170859_20190525T171218_T14TQM:   0%|          |0/390 tiles [00:00<?]

...

Download completed: ./Lab4_Assignment2/AOI_Nebe/20190525T170859_20190525T171218_T15TUE/20190525T170859_20190525T171218_T15TUE.tif

Processing AOI: AOI_NorthAL
Matching images found: 2

Downloading: 20190525T161901_20190525T163308_T16SFC


20190525T161901_20190525T163308_T16SFC:   0%|          |0/330 tiles [00:00<?]

Download completed: ./Lab4_Assignment2/AOI_NorthAL/20190525T161901_20190525T163308_T16SFC/20190525T161901_20190525T163308_T16SFC.tif

Downloading: 20190525T161901_20190525T163308_T16SFD


20190525T161901_20190525T163308_T16SFD:   0%|          |0/330 tiles [00:00<?]

Download completed: ./Lab4_Assignment2/AOI_NorthAL/20190525T161901_20190525T163308_T16SFD/20190525T161901_20190525T163308_T16SFD.tif

Assignment 2 completed!
Output folder: ./Lab4_Assignment2
